# s-SNOM spectra without the app

Define a frequency grid, tip, sample and reference, then calculate all selected
harmonics in **one engine call**. This example uses bulk silica normalized to
gold, followed by a 30 nm silica film on silicon and a two-film stack. No Streamlit session
is needed. Basic Python and NumPy familiarity is enough.

From the repository root, install and open the notebook using Python 3.11+:

```bash
python -m pip install -e . jupyterlab
python -m jupyterlab examples/engine_spectra.ipynb
```

Use a kernel from the same Python environment. Run the cells from top to bottom:
1. Define the setup.
2. Calculate and plot the bulk harmonics.
3. Calculate and plot a finite layer.
4. Calculate and plot a multilayer stack.
5. Combine catalog, custom and constant materials, uniaxial films and forced FIM.
6. Adapt the materials and inspect the returned arrays.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from snom_calculator.bulk import calculate_bulk_harmonics
from snom_calculator.layer import calculate_layer_harmonics
from snom_calculator.materials import catalog_material, constant_material
from snom_calculator.models import TipParameters, NumericalSettings, FarFieldSettings

## 1. Define the setup

The spectral coordinate is **wavenumber in cm⁻¹**; all lengths are in **nm**.
Material spectra must be evaluated on the same grid used by the engine.
`catalog_material` loads the same catalog used by the app (`aSiO2`, `Au`, `Si`).
The default tip is an approximately perfect conductor with constant ε.

In [ ]:
frequency_cm1 = np.linspace(900.0, 1300.0, 81)
harmonics = (2, 3, 4)

tip = TipParameters(
    radius_nm=25.0,
    length_nm=600.0,
    amplitude_nm=40.0,
    sample_gap_nm=2.0,
    reference_gap_nm=2.0,
    epsilon=-1e10 + 0j,
)
sample = catalog_material("aSiO2", frequency_cm1)
reference = catalog_material("Au", frequency_cm1)

numerical = NumericalSettings()
far_field = FarFieldSettings(enabled=True, angle_deg=60.0)

## 2. Calculate the bulk harmonics

The engine returns a dictionary keyed by harmonic order. Each `SpectrumResult`
contains the complex contrast `sample_harmonic / reference_harmonic`, its amplitude,
and its unwrapped phase in radians. The phase convention shifts a negative starting
phase up by 2π. All harmonics share the same underlying response calculation.

In [ ]:
bulk_results = calculate_bulk_harmonics(
    frequency_cm1,
    tip=tip,
    sample=sample,
    reference=reference,
    harmonics=harmonics,
    numerical=numerical,
    far_field=far_field,
)

### Plot amplitude and phase

This small plotting helper works with either engine's results. Colors match
between the amplitude and phase panels; click a legend item to toggle a harmonic.

In [ ]:
def plot_harmonics(results, title):
    figure = make_subplots(rows=1, cols=2, subplot_titles=("Amplitude", "Phase"))
    colors = ("#0072B2", "#D55E00", "#009E73", "#CC79A7")
    for index, (harmonic, result) in enumerate(results.items()):
        line = dict(color=colors[index % len(colors)])
        for column, values in ((1, result.amplitude), (2, result.phase_rad)):
            figure.add_trace(go.Scatter(
                x=result.frequency_cm1, y=values, mode="lines",
                name=f"n = {harmonic}", legendgroup=str(harmonic),
                showlegend=column == 1, line=line,
            ), row=1, col=column)
    figure.update_xaxes(title_text="Wavenumber (cm⁻¹)")
    figure.update_yaxes(title_text="Normalized amplitude", row=1, col=1)
    figure.update_yaxes(title_text="Phase (rad)", row=1, col=2)
    figure.update_layout(title=title, template="plotly_white", height=420)
    return figure


plot_harmonics(bulk_results, "Bulk silica / gold").show()

## 3. Calculate a finite layer

Reuse the tip, frequency grid and reference. The sample material now describes
the film; add its thickness and the semi-infinite substrate. Keep `workers=1`
(the default) for simple notebook execution. First execution may take longer
while numerical functions compile and geometry caches warm up.

In [ ]:
substrate = catalog_material("Si", frequency_cm1)
thickness_nm = 30.0

layer_results = calculate_layer_harmonics(
    frequency_cm1,
    tip=tip,
    layer=sample,
    substrate=substrate,
    thickness_nm=thickness_nm,
    reference=reference,
    harmonics=harmonics,
    numerical=numerical,
    far_field=far_field,
)
plot_harmonics(layer_results, "30 nm silica on silicon / gold").show()

## 4. Calculate a multilayer stack

For several films use `calculate_stack_harmonics` from `snom_calculator.layer`.
`layers` is a list of `(material, thickness_nm)` pairs ordered **from the tip toward
the substrate**; the substrate stays semi-infinite. The other arguments and the
returned results match the single layer. Here a 10 nm silica film covers a 20 nm
film of constant ε on silicon. Zero-thickness films are dropped and identical
neighbours are merged; an empty list gives the bulk substrate.

In [ ]:
from snom_calculator.layer import calculate_stack_harmonics

film = constant_material(4.0 + 0.2j, frequency_cm1, "Constant film")

stack_results = calculate_stack_harmonics(
    frequency_cm1,
    layers=[(sample, 10.0), (film, 20.0)],
    substrate=substrate,
    reference=reference,
    tip=tip,
    numerical=numerical,
    harmonics=harmonics,
    far_field=far_field,
)
plot_harmonics(stack_results, "10 nm silica / 20 nm film on silicon / gold").show()

### Force the Fourier integral method (FIM)

`method="auto"` (the default) picks the film coupling itself: image charges (`"gicm"`) for
isotropic films, FIM otherwise. Pass `method="fim"` to integrate the exact film reflection
on a rotated momentum contour instead; it needs no image-charge count, so it also suits
films with guided modes near the real axis. `"gicm"` and `"real-axis"` are the other
choices. Both calls accept the same arguments, so comparing methods is a one-word change.
FIM is slower; the printed difference should be small when the image-charge result is converged.

In [ ]:
fim_results = calculate_stack_harmonics(
    frequency_cm1,
    layers=[(sample, 10.0), (film, 20.0)],
    substrate=substrate,
    reference=reference,
    tip=tip,
    numerical=numerical,
    harmonics=harmonics,
    far_field=far_field,
    method="fim",
)
plot_harmonics(fim_results, "Same stack, method='fim'").show()
for harmonic in harmonics:
    difference = np.abs(fim_results[harmonic].contrast / stack_results[harmonic].contrast - 1).max()
    print(f"n = {harmonic}: max relative difference FIM vs auto = {difference:.2e}")

## 5. Combined examples

Three short workflows that mix the pieces above. They use the **hBN** Reststrahlen
band (1370–1610 cm⁻¹), where the in-plane and surface-normal permittivities differ strongly.
Catalog hBN is **uniaxial**: `parallel` is the surface-normal εzz and `perpendicular` the in-plane εxx.
Uniaxial films automatically use FIM under `method="auto"`.

In [ ]:
freq_hbn = np.linspace(1350.0, 1650.0, 41)
hbn = catalog_material("hBN", freq_hbn)
si = catalog_material("Si", freq_hbn)
gold = catalog_material("Au", freq_hbn)
print(f"hBN uniaxial: {not hbn.isotropic}; Re ε(in-plane) range "
      f"{hbn.perpendicular.real.min():.1f} to {hbn.perpendicular.real.max():.1f}; "
      f"Re ε(normal) {hbn.parallel.real.min():.1f} to {hbn.parallel.real.max():.1f}")

def compare(results_by_label, harmonic, quantity="amplitude"):
    """Overlay one harmonic of several calculations."""
    figure = go.Figure()
    for label, results in results_by_label.items():
        result = results[harmonic]
        figure.add_trace(go.Scatter(x=result.frequency_cm1, y=getattr(result, quantity),
                                    mode="lines", name=label))
    figure.update_layout(template="plotly_white", height=380, xaxis_title="Wavenumber (cm⁻¹)",
                         yaxis_title=quantity, title=f"n = {harmonic}")
    return figure

### Example A: bulk uniaxial hBN, library reference vs constant reference

*Combines:* library (catalog) materials, a uniaxial sample, bulk engine, a constant material.
The same sample is normalized once to gold and once to a constant ε. Only the reference changes,
so the difference shows how the choice of reference rescales the contrast.

In [ ]:
constant_ref = constant_material(-200 + 60j, freq_hbn, "Constant metal")
common = dict(tip=tip, harmonics=(3,), numerical=numerical, far_field=far_field)

bulk_gold = calculate_bulk_harmonics(freq_hbn, sample=hbn, reference=gold, **common)
bulk_constant = calculate_bulk_harmonics(freq_hbn, sample=hbn, reference=constant_ref, **common)
compare({"hBN / Au": bulk_gold, "hBN / constant": bulk_constant}, 3).show()

### Example B: a custom material saved in a library, used in a forced-FIM stack

*Combines:* custom material (Lorentz in-plane axes, **constant** surface-normal axis, Drude term),
the material library, a multilayer stack, constant spacer, catalog hBN and `method="fim"` and `"real-axis"`.

A definition is a plain dictionary with one entry per tensor axis (`xx`, `yy` in-plane, `zz` normal);
each axis is `weak_lorentz`, `strong_lorentz`, `constant` or `table`. The optional `drude` term adds
a free-carrier response to every listed axis. `SessionMaterialLibrary` is an in-memory
library; `JsonMaterialLibrary` and `MaterialLibrary` keep the same definitions on disk.
`custom_material` evaluates a saved definition on your frequency grid.

This stack contains uniaxial films, so `method="auto"` already selects FIM and forcing it reproduces
the same numbers. The `"real-axis"` integral is an independent cross-check of the same physics.

In [ ]:
from snom_calculator.material_library import SessionMaterialLibrary
from snom_calculator.materials import custom_material

lorentz = {"source_type": "weak_lorentz", "eps_inf": 3.0,
           "oscillators": [{"strength": 0.6, "w0_cm1": 1450.0, "gamma_cm1": 15.0}]}
normal = {"source_type": "constant", "eps_real": 2.5, "eps_imag": 0.0}
doped = {"name": "Doped oxide", "tensor_mode": "anisotropic_diagonal",
         "axes": {"xx": lorentz, "yy": lorentz, "zz": normal},
         "drude": {axis: {"wp_cm1": 500.0, "gamma_cm1": 100.0} for axis in ("xx", "yy", "zz")}}

library = SessionMaterialLibrary({})
library.save(doped)
print("Library contains:", library.names())
doped_film = custom_material(library.load("Doped oxide"), freq_hbn)

spacer = constant_material(2.1 + 0j, freq_hbn, "Spacer")
layers = [(spacer, 5.0), (doped_film, 20.0), (hbn, 30.0)]  # tip -> substrate
stack_args = dict(layers=layers, substrate=si, reference=gold, tip=tip, harmonics=(2, 3),
                  numerical=numerical, far_field=far_field)

stack_auto = calculate_stack_harmonics(freq_hbn, **stack_args)
stack_fim = calculate_stack_harmonics(freq_hbn, method="fim", **stack_args)
stack_real = calculate_stack_harmonics(freq_hbn, method="real-axis", **stack_args)
compare({"auto": stack_auto, "forced FIM": stack_fim, "real-axis": stack_real}, 3).show()
for harmonic in (2, 3):
    for name, other in (("FIM", stack_fim), ("real-axis", stack_real)):
        difference = np.abs(other[harmonic].contrast / stack_auto[harmonic].contrast - 1).max()
        print(f"n = {harmonic}: max relative difference {name} vs auto = {difference:.2e}")

### Example C: film thickness series with a dielectric tip

*Combines:* a layer series, uniaxial hBN film, a constant (dielectric) tip, a constant reference
and the harmonic comparison. The tip is a constant complex ε as well: here a silicon-like tip
instead of the near-perfect conductor. Each curve is one single-layer call.

In [ ]:
dielectric_tip = TipParameters(radius_nm=25.0, length_nm=600.0, amplitude_nm=40.0,
                               sample_gap_nm=2.0, reference_gap_nm=2.0, epsilon=11.7 + 0j)
series = {
    f"{thickness:g} nm": calculate_layer_harmonics(
        freq_hbn, tip=dielectric_tip, layer=hbn, substrate=si, thickness_nm=thickness,
        reference=constant_ref, harmonics=(3,), numerical=numerical, far_field=far_field,
    )
    for thickness in (10.0, 30.0, 60.0)
}
compare(series, 3, "amplitude").show()
compare(series, 3, "phase_rad").show()

## 6. Inspect and adapt

For example, `bulk_results[3].amplitude` and `bulk_results[3].phase_rad` are NumPy
arrays with one value per wavenumber. Raw complex signals are available as
`.sample_harmonic`, `.reference_harmonic` and `.contrast`.

The checks below catch empty, misaligned or non-finite results after edits.

In [ ]:
for results in (bulk_results, layer_results, stack_results, fim_results):
    assert tuple(results) == harmonics
    freq = frequency_cm1
    for result in results.values():
        np.testing.assert_array_equal(result.frequency_cm1, frequency_cm1)
        for values in (result.sample_harmonic, result.reference_harmonic,
                       result.contrast, result.amplitude, result.phase_rad):
            assert values.shape == frequency_cm1.shape
            assert np.isfinite(values).all()
print(f"All engines returned {len(harmonics)} harmonics at {frequency_cm1.size} wavenumbers.")

### Other material definitions

Use `constant_material(12 + 0.1j, frequency_cm1, "My dielectric")` for constant ε.
For your own dispersive model, create a `MaterialSpectrum` (from `snom_calculator.materials`) from complex arrays:
`MaterialSpectrum(parallel=epsilon, perpendicular=epsilon, label="My sample")`.
Here `parallel` means **surface-normal εzz**, and `perpendicular` means **in-plane εxx**;
use different arrays for a surface-normal uniaxial material.
To read a table, use `parse_material_table` from `snom_calculator.materials`.

The tip uses constant complex permittivity. Changing the frequency grid
requires rebuilding all sample and reference material spectra.

### Try it

Replace the reference with constant ε in the next cell and recalculate. Then try
changing the frequency range or film thickness above and rerun all cells.
`NumericalSettings()` is a starting point: check convergence by increasing `M`,
`N`, `quadQ` and (for isotropic layers) `K` near resonances. `M` must exceed the
highest harmonic and `quadQ` must be at least `N`. Use
`FarFieldSettings(enabled=False)` to omit the far-field multiplier.
These direct calls do not run the app's automatic convergence checks, gap
averaging or spectral-resolution broadening.

In [ ]:
constant_reference = constant_material(-5000 + 800j, frequency_cm1, "Constant reference")
alternative_results = calculate_bulk_harmonics(
    frequency_cm1,
    tip=tip,
    sample=sample,
    reference=constant_reference,
    harmonics=harmonics,
    numerical=numerical,
    far_field=far_field,
)
plot_harmonics(alternative_results, "Bulk silica / constant reference").show()